# 🔥 Single-Keypoint Fire Base Grounding via End-to-End YOLO26-Pose

## 1. Bối cảnh & Điểm Mới Nghiên Cứu (Research Gap & Novelty)
- **Vấn đề của V2 / V3 (Heatmap + SoftArgmax):**
  - Heatmap dự đoán trên toàn ảnh không có ràng buộc bounding box -> Khi gặp ảnh thực tế ngoài trời, đêm tối, ánh đèn phản chiếu, SoftArgmax bị **đa đỉnh (multimodal ambiguity)** kéo điểm chân lửa trôi ra giữa lòng đường tối hoặc xe nôi.
  - Chi phí bộ nhớ và tính toán lớn ($64 \times 64$ ma trận xác suất chỉ để lấy 2 con số $(x, y)$), dễ bị sai số làm tròn (quantization error).
- **Giải pháp Đột Phá: Single-Keypoint YOLO26-Pose (`yolo26n-pose`):**
  1. **Khóa chặt không gian (Spatial Box Constraint):** Điểm chân lửa tiếp xúc mặt sàn $(k_x, k_y)$ được neo trực tiếp vào hộp ngọn lửa (Bounding Box). Tuyệt đối **không thể trôi ra ngoài** sang người đi đường hay vỉa hè.
  2. **Siêu nhẹ & Siêu nhanh (Edge Real-time):** Model `yolo26n-pose` chỉ có **~2.7M parameters** (nhẹ hơn V3 7.5M hơn một nửa!), kiến trúc **Native End-to-End (NMS-free)** cho độ trễ chỉ ~3ms, hoàn toàn phù hợp nhúng vào Drone và Robot dập lửa tự hành.
  3. **Tính Mới Tuyệt Đối (Novelty):** 99% các nghiên cứu về cháy chỉ phát hiện hộp ngọn lửa (Detection) hoặc phân vùng khói (Segmentation). Đây là công trình đầu tiên xây dựng pipeline chuẩn **Single-Keypoint Pose Estimation (`kpt_shape: [1, 2]`)** chuyên biệt để robot định vị chính xác vị trí tiếp xúc sàn của đám cháy!

In [ ]:
# Cell 1: KIỂM TRA MÔI TRƯỜNG & PHẦN CỨNG GPU
import torch
import ultralytics
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'✅ PyTorch Version: {torch.__version__}')
print(f'✅ Ultralytics Version: {ultralytics.__version__}')
print(f'✅ CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🚀 GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ Đang chạy trên CPU. Khuyến nghị kích hoạt GPU để train nhanh hơn.')


In [ ]:
# Cell 2: KIỂM TRA DATASET ĐỊNH VỊ CHÂN LỬA (YOLO-POSE FORMAT)
# Cấu trúc nhãn 7 cột cho 1 Keypoint: <class_id> <cx> <cy> <w> <h> <kpt_x> <kpt_y>
import json, glob
from pathlib import Path

# Tự động tìm thư mục dataset_fire_pose ở root hoặc subfolder
candidate_paths = [Path('dataset_fire_pose'), Path('../dataset_fire_pose')]
DATASET_ROOT = next((p for p in candidate_paths if p.exists()), Path('dataset_fire_pose'))
print(f'📂 Dataset Directory: {DATASET_ROOT.resolve()}')

# Kiểm tra số lượng ảnh và nhãn theo từng split
for split in ['train', 'val', 'test']:
    imgs = list((DATASET_ROOT / 'images' / split).glob('*.jpg'))
    lbls = list((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    fire_cnt = sum(1 for lp in lbls if lp.stat().st_size > 0)
    print(f'  • Split [{split.upper()}]: {len(imgs)} ảnh | {fire_cnt} ảnh có lửa | {len(lbls) - fire_cnt} ảnh nền (Hard Negative)')


In [ ]:
# Cell 3: TẠO FILE CẤU HÌNH DATASET YAML (kpt_shape: [1, 2])
import yaml
from pathlib import Path

candidate_paths = [Path('dataset_fire_pose'), Path('../dataset_fire_pose')]
DATASET_ROOT = next((p for p in candidate_paths if p.exists()), Path('dataset_fire_pose'))

yaml_data = {
    'path': str(DATASET_ROOT.resolve()).replace('\\', '/'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'kpt_shape': [1, 2],  # 1 keypoint: (x_norm, y_norm)
    'flip_idx': [],        # Không lật keypoint đối xứng
    'names': {
        0: 'fire'
    }
}

yaml_path = DATASET_ROOT / 'fire_base_pose.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f'✅ Đã lưu cấu hình: {yaml_path.resolve()}')
with open(yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())


In [ ]:
# Cell 4: KHỞI TẠO MÔ HÌNH YOLO26-POSE (yolo26n-pose.pt)
from pathlib import Path
from ultralytics import YOLO

# Tìm file trọng số gốc yolo26n-pose.pt
weights_candidates = [Path('yolo26n-pose.pt'), Path('../yolo26n-pose.pt')]
base_weight = next((p for p in weights_candidates if p.exists()), 'yolo26n-pose.pt')

model = YOLO(str(base_weight))
total_params = sum(p.numel() for p in model.model.parameters())
print(f'🚀 Model Name: YOLO26n-Pose')
print(f'📊 Pretrained Params: {total_params:,} ({total_params/1e6:.2f}M)')
print(f'⚡ Kiến trúc: Native End-to-End NMS-Free Head + Flow Pose Estimator')


In [ ]:
# Cell 5: VÒNG LẶP HUẤN LUYỆN (TRAINING PIPELINE)
# Tối ưu hóa siêu tham số cho GPU RTX 3060 trên hệ điều hành Windows
import torch
from pathlib import Path
from ultralytics import YOLO

candidate_paths = [Path('dataset_fire_pose'), Path('../dataset_fire_pose')]
DATASET_ROOT = next((p for p in candidate_paths if p.exists()), Path('dataset_fire_pose'))
yaml_path = DATASET_ROOT / 'fire_base_pose.yaml'

weights_candidates = [Path('yolo26n-pose.pt'), Path('../yolo26n-pose.pt')]
base_weight = next((p for p in weights_candidates if p.exists()), 'yolo26n-pose.pt')

if 'model' not in locals():
    model = YOLO(str(base_weight))

device_target = 0 if torch.cuda.is_available() else 'cpu'
print(f'🔥 Bắt đầu huấn luyện YOLO26-Pose trên thiết bị: {device_target}...')

# Trên Windows trong Jupyter, đặt workers=0 để tránh xung đột tiến trình
results = model.train(
    data=str(yaml_path.resolve()),
    epochs=25,              # 25 epochs hội tụ tối ưu
    imgsz=384,              # Kích thước 384x384 bắt nét chi tiết chân lửa
    batch=16,               # Batch 16 vừa vặn VRAM RTX 3060
    device=device_target,
    workers=0,              # workers=0 an toàn 100% trên Windows
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    mosaic=1.0,             # Ghép 4 ảnh tăng sức chịu đựng nhiễu
    mixup=0.1,              # Trộn ảnh giả lập khói mờ
    project='fire_pose_runs',
    name='yolo26n_fire_base',
    save=True,
    plots=True,
    verbose=True
)

print('🎉 HUẤN LUYỆN HOÀN TẤT! File trọng số đã được lưu tại: fire_pose_runs/yolo26n_fire_base/weights/best.pt')


In [ ]:
# Cell 6: ĐÁNH GIÁ ĐỊNH LƯỢNG (VALIDATION & BENCHMARK METRICS)
# Đánh giá trên tập kiểm thử Test Set (1,300 ảnh chuẩn HomeFire)
import torch
from pathlib import Path
from ultralytics import YOLO

candidate_paths = [Path('dataset_fire_pose'), Path('../dataset_fire_pose')]
DATASET_ROOT = next((p for p in candidate_paths if p.exists()), Path('dataset_fire_pose'))
yaml_path = DATASET_ROOT / 'fire_base_pose.yaml'

# Tìm checkpoint đã huấn luyện ở các vị trí có thể có
ckpt_candidates = [
    Path('fire_pose_runs/yolo26n_fire_base/weights/best.pt'),
    Path('../fire_pose_runs/yolo26n_fire_base/weights/best.pt'),
    Path('FireGrounder-V3-main/fire_pose_runs/yolo26n_fire_base/weights/best.pt')
]
best_ckpt = next((p for p in ckpt_candidates if p.exists()), None)

if best_ckpt is None:
    print('='*70)
    print('⚠️ THÔNG BÁO: Chưa tìm thấy file trọng số "best.pt"!')
    print('👉 Lý do: Cell 5 (Huấn luyện) chưa chạy xong nên chưa sinh ra file này.')
    print('👉 Hãy bấm chạy Cell 5 (mất khoảng vài phút), sau đó bấm lại Cell 6 nhé!')
    print('='*70)
else:
    print(f'📦 Đang tải checkpoint: {best_ckpt.resolve()}')
    eval_model = YOLO(str(best_ckpt))
    device_target = 0 if torch.cuda.is_available() else 'cpu'
    metrics = eval_model.val(data=str(yaml_path.resolve()), split='test', imgsz=384, device=device_target)
    
    print('\n' + '='*60)
    print('📊 BÁO CÁO ĐÁNH GIÁ ĐỊNH LƯỢNG YOLO26-POSE TRÊN TEST SET (1,300 ẢNH)')
    print('='*60)
    print(f'🔥 Bounding Box mAP50:      {metrics.box.map50:.4f}')
    print(f'🔥 Bounding Box mAP50-95:   {metrics.box.map:.4f}')
    print(f'📍 Fire Base Pose mAP50:    {metrics.pose.map50:.4f}')
    print(f'📍 Fire Base Pose mAP50-95: {metrics.pose.map:.4f}')
    print('='*60)


In [ ]:
# Cell 7: THỬ NGHIỆM THỰC CHIẾN TRÊN ẢNH NGOẠI CẢNH (MỖI HÀNG 2 ẢNH TO RÕ)
# Thử thách Out-Of-Distribution: đường phố ban đêm, đèn đường, người đi bộ, khói mờ
import random, cv2, torch
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

ckpt_candidates = [
    Path('fire_pose_runs/yolo26n_fire_base/weights/best.pt'),
    Path('../fire_pose_runs/yolo26n_fire_base/weights/best.pt'),
    Path('FireGrounder-V3-main/fire_pose_runs/yolo26n_fire_base/weights/best.pt'),
    Path('runs/pose/fire_pose_runs/yolo26n_fire_base-2/weights/best.pt'),
]
best_ckpt = next((p for p in ckpt_candidates if p.exists()), None)

if best_ckpt is None:
    print('⚠️ Cần chạy xong Cell 5 để có checkpoint "best.pt" trước khi chạy thử nghiệm.')
else:
    print(f'📦 Đang dùng checkpoint: {best_ckpt.resolve()}')
    eval_model = YOLO(str(best_ckpt))
    sample_candidates = [Path('fire-samples'), Path('../fire-samples')]
    sample_dir = next((p for p in sample_candidates if p.exists()), Path('fire-samples'))
    sample_imgs = list(sample_dir.glob('*.png')) + list(sample_dir.glob('*.jpg'))
    print(f'Tổng số ảnh ngoại cảnh fire-samples: {len(sample_imgs)}')
    
    # =========================================================================
    # CẤU HÌNH HIỂN THỊ: Mỗi hàng 2 ảnh to rõ nét (kích thước lớn dễ quan sát)
    # =========================================================================
    num_images = 10   # Số lượng ảnh hiển thị (có thể đổi thành 6, 8, 12 tùy thích)
    ncols = 2        # Cố định mỗi hàng 2 ảnh
    nrows = (num_images + ncols - 1) // ncols
    
    random.seed(42)
    chosen_imgs = random.sample(sample_imgs, min(num_images, len(sample_imgs)))
    
    # Khổ ảnh rộng 16 inch, mỗi hàng cao 6 inch -> ảnh to gấp đôi, siêu nét
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 6 * nrows))
    axes = axes.flatten()
    device_target = 0 if torch.cuda.is_available() else 'cpu'
    
    for idx, img_p in enumerate(chosen_imgs):
        img_bgr = cv2.imread(str(img_p))
        if img_bgr is None:
            continue
        
        # Chạy inference với YOLO26-Pose
        res = eval_model.predict(img_bgr, conf=0.25, imgsz=384, device=device_target, verbose=False)[0]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        has_det = False
        if len(res.boxes) > 0:
            has_det = True
            boxes = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            kpts = res.keypoints.xy.cpu().numpy() if res.keypoints is not None else []
            
            for b_i, box in enumerate(boxes):
                x1, y1, x2, y2 = [int(v) for v in box]
                cf = confs[b_i]
                # Bounding Box nét dày (xanh lục)
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(img_rgb, f'Fire {cf:.2f}', (x1, max(25, y1 - 8)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                # Keypoint chân gốc lửa to rõ (chữ thập đỏ tâm vàng)
                if len(kpts) > b_i:
                    kx, ky = int(kpts[b_i][0][0]), int(kpts[b_i][0][1])
                    if kx > 0 and ky > 0:
                        cv2.circle(img_rgb, (kx, ky), 10, (255, 255, 0), -1)
                        cv2.circle(img_rgb, (kx, ky), 4, (255, 0, 0), -1)
                        cv2.line(img_rgb, (kx - 15, ky), (kx + 15, ky), (255, 0, 0), 2)
                        cv2.line(img_rgb, (kx, ky - 15), (kx, ky + 15), (255, 0, 0), 2)
        
        axes[idx].imshow(img_rgb)
        title_color = 'green' if has_det else 'gray'
        status_txt = 'DETECTED' if has_det else 'NO FIRE'
        axes[idx].set_title(f'{img_p.name} [{status_txt}]', fontsize=12, fontweight='bold', color=title_color)
        axes[idx].axis('off')
    
    # Ẩn các ô trống nếu có
    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.savefig('preview_yolo26_fire_samples.png', dpi=150)
    plt.show()
    print('✅ Đã hiển thị mỗi hàng 2 ảnh to rõ và lưu vào preview_yolo26_fire_samples.png!')
